# Gemma Security Project Notebook

This notebook is the starting guide for the `gemma-security` project. It explains the project structure, the code files, the dataset format, and the first training/inference workflow.

We are using **Gemma 2** because it is small enough for early experimentation while still being strong enough for useful cybersecurity text tasks.

## 1. What We Are Building

The first version of this project is a cybersecurity text classifier.

Input example:

> A user received an email asking them to reset their bank password through an unknown link.

Expected output example:

> `phishing`

Later, this project can grow into a security assistant that summarizes alerts, explains vulnerabilities, ranks risk, or suggests response steps.

## 2. Project Structure

```text
gemma-security/
├── notebooks/                 # Learning and experimentation notebooks
├── src/gemma_security/         # Reusable Python project code
├── data/raw/                   # Original CSV data
├── data/processed/             # Cleaned or transformed data
├── models/                     # Saved trained models
├── outputs/                    # Logs, reports, predictions
├── requirements.txt            # Python packages
└── README.md                   # Project overview
```

The notebook is for understanding and experimenting. The `src/` files are for reusable code that can later become a real app, API, or training pipeline.

## 3. Install Dependencies

Run this from the project root, not from inside the notebook folder:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Gemma models may require Hugging Face access approval. After accepting the model terms, sign in:

```bash
huggingface-cli login
```

In [ ]:
# This cell lets the notebook import code from the project src/ folder.
# If you run the notebook from another location, update PROJECT_ROOT.

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

## 4. Configuration

The file `src/gemma_security/config.py` keeps the important project settings in one place.

This is useful because you do not want model names, paths, and labels copied across many files.

In [ ]:
from gemma_security.config import LABELS, MODEL_NAME, RAW_DATA_PATH, OUTPUT_MODEL_DIR

print("Model:", MODEL_NAME)
print("Labels:", LABELS)
print("Raw data path:", RAW_DATA_PATH)
print("Output model path:", OUTPUT_MODEL_DIR)

## 5. Dataset Format

The first dataset should be a CSV file at `data/raw/security_examples.csv`.

Required columns:

- `text`: the security event, report, log summary, or alert text
- `label`: the correct category

Starter labels:

- `phishing`
- `malware`
- `vulnerability`
- `bruteforce`
- `benign`

In [ ]:
# This creates a tiny example dataset only if you do not already have one.
# Replace this sample with real project data when you are ready.

import pandas as pd

RAW_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.exists():
    sample_rows = [
        {
            "text": "A user received an email asking them to reset a bank password through an unknown link.",
            "label": "phishing",
        },
        {
            "text": "Endpoint protection found a suspicious executable trying to persist after reboot.",
            "label": "malware",
        },
        {
            "text": "A dependency scan reports a critical CVE in an outdated package.",
            "label": "vulnerability",
        },
        {
            "text": "SSH logs show hundreds of failed login attempts from the same IP address.",
            "label": "bruteforce",
        },
        {
            "text": "The weekly backup job completed successfully with no unusual activity.",
            "label": "benign",
        },
    ]
    pd.DataFrame(sample_rows).to_csv(RAW_DATA_PATH, index=False)

pd.read_csv(RAW_DATA_PATH)

## 6. Prompt Design

The model should see inputs in a consistent style. The helper `build_security_prompt()` wraps raw text into a clear instruction.

Raw text is the security event. The prompt tells Gemma what job to do with that text.

In [ ]:
from gemma_security.data import build_security_prompt

example_text = "A login page clone is asking employees to enter their company password."
print(build_security_prompt(example_text))

## 7. Loading And Validating Data

`load_security_dataframe()` checks that the CSV has the required columns and that labels match the allowed label list.

This protects the project from silent mistakes, like spelling `phishing` as `phisihng`.

In [ ]:
from gemma_security.data import load_security_dataframe

security_df = load_security_dataframe()
security_df.head()

## 8. Turning Data Into A Training Dataset

Hugging Face training uses `Dataset` objects. The project helper `build_dataset_dict()` creates a train/test split.

The train split teaches the model. The test split gives us a small check on how the model performs on examples it did not train on.

In [ ]:
from gemma_security.data import build_dataset_dict

dataset_dict = build_dataset_dict()
dataset_dict

## 9. Loading Gemma 2

`src/gemma_security/model.py` contains two helpers:

- `load_tokenizer()` loads the text tokenizer
- `load_classifier_model()` loads Gemma 2 with a classification head

The tokenizer turns text into numbers. The model learns patterns in those numbers.

In [ ]:
# This cell downloads Gemma 2, so it may take time and may require Hugging Face access.
# Uncomment when your environment is ready.

# from gemma_security.model import load_tokenizer, load_classifier_model
# tokenizer = load_tokenizer()
# model = load_classifier_model()
# print(type(tokenizer))
# print(type(model))

## 10. Tokenization

Before training, prompts must be tokenized. The training script does this with `tokenize_dataset()`.

Tokenization also truncates very long text so it fits within `MAX_LENGTH` from the config file.

In [ ]:
# Example tokenization flow. Uncomment after loading the tokenizer above.

# from gemma_security.train import tokenize_dataset
# tokenized_dataset = tokenize_dataset(dataset_dict, tokenizer)
# tokenized_dataset

## 11. Training

The training entry point lives in `src/gemma_security/train.py`.

For real training, use more than the tiny sample dataset created above. A tiny dataset is useful for testing the pipeline, but it cannot teach the model enough to become reliable.

In [ ]:
# Run this only after you have Hugging Face access and enough data.

# from gemma_security.train import train
# train()

## 12. Inference

After training, the model is saved under `models/gemma-security-classifier`.

`predict_security_label()` loads the trained model and returns a label plus confidence score.

In [ ]:
# Uncomment after a trained model exists in models/gemma-security-classifier.

# from gemma_security.inference import predict_security_label
# predict_security_label("Suspicious email asks an employee to verify their account password.")

## 13. Codebase Summary

| File | Purpose |
| --- | --- |
| `config.py` | Stores labels, paths, model name, and training settings |
| `data.py` | Loads CSV data, validates labels, builds prompts, creates train/test datasets |
| `model.py` | Loads the Gemma tokenizer and classifier model |
| `train.py` | Tokenizes data, trains the model, saves the result |
| `inference.py` | Loads a trained model and predicts labels for new text |

The notebook shows the same flow in learning mode. The scripts let you reuse the same logic outside the notebook.

## 14. Next Build Steps

1. Add a real CSV dataset at `data/raw/security_examples.csv`.
2. Confirm the label list in `config.py` matches your actual security categories.
3. Run the notebook cells through data loading and validation.
4. Log in to Hugging Face and load Gemma 2.
5. Run a small training test.
6. Improve the dataset and training settings based on results.

This notebook should keep growing as the project grows.